# 02 — Preprocessing (Phase 2)

Objective: freeze `feature_list.json`, clean rows, split one-class, and fit the
StandardScaler on **benign-train only** (DRD D-05–D-13, ERD C1–C3).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

ROOT = D:\Study\Sem 5\AIML lab\pca-network-anomaly-detection-clone


In [2]:
import numpy as np
import pandas as pd

from src import SEED
from src.data_loader import load_dataset
from src.feature_selection import align_features, select_features
from src.preprocessing import Preprocessor, clean_frame, save_processed, split_one_class

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
PROC.mkdir(parents=True, exist_ok=True)

A = load_dataset(RAW / "synthetic_A.csv")
B = load_dataset(RAW / "synthetic_B.csv")
print(A.shape, B.shape)

(14000, 44) (7200, 42)


In [3]:
keptA, repA = select_features(A)
keptB, _ = select_features(B)
repA.to_csv(TAB / "feature_report.csv", index=False)
print(f"keptA={len(keptA)} keptB={len(keptB)}")
print(repA[repA.action == "drop"][["feature", "reason"]].to_string() if (repA.action == "drop").any() else "nothing dropped")

common, align = align_features(keptA, keptB)
align.to_csv(TAB / "feature_alignment.csv", index=False)
print("common:", len(common), "| dropped by B:", align[align.status == "only_A"].feature.tolist())

keptA=40 keptB=38
           feature                     reason
40           Label  excluded/leaky identifier
41  AttackCategory  excluded/leaky identifier
42         flow_id  excluded/leaky identifier
43      dataset_id  excluded/leaky identifier
common: 38 | dropped by B: ['urg_flag_cnt', 'idle_max']


In [4]:
Ac, info = clean_frame(A, keptA)
print("cleaning:", info)
sp = split_one_class(Ac, seed=SEED)
print({k: len(v) for k, v in sp.items()})

# scaler + medians fit on BENIGN-TRAIN ONLY (ERD C1); transform-only below.
pp40 = Preprocessor(keptA).fit(sp["train_benign"])
pp40.save(PROC / "scaler.pkl")
pp38 = Preprocessor(common).fit(sp["train_benign"])
pp38.save(PROC / "scaler_common.pkl")

Xtr = pp40.transform(sp["train_benign"])
Xb = pp40.transform(sp["test_benign"])
Xa = pp40.transform(sp["test_attack"])
print("scaled train mean≈%.2e std≈%.3f" % (Xtr.mean(), Xtr.std()))

from src import BENIGN_LABEL, CATEGORY_COL
yte = np.array([0] * len(Xb) + [1] * len(Xa))
cats = np.array([BENIGN_LABEL] * len(Xb) + sp["test_attack"][CATEGORY_COL].tolist())
save_processed(PROC / "processed_A.npz", X_train=Xtr, X_test_benign=Xb,
               X_test_attack=Xa, y_test=yte, test_cats=cats,
               meta={"seed": SEED, "n_train_benign": len(Xtr), "features": len(keptA)})
print("saved scaler.pkl, scaler_common.pkl, feature_list.json, processed_A.npz")

cleaning: {'rows_before': 14000, 'rows_dropped': 0}
{'train_benign': 5600, 'test_benign': 2400, 'test_attack': 6000}


scaled train mean≈-3.23e-17 std≈1.000


saved scaler.pkl, scaler_common.pkl, feature_list.json, processed_A.npz


## Checkpoint — Phase 2 verdict

- DRD B7: `feature_list.json` + `feature_report.csv` + `scaler.pkl` lineage frozen.
- Scaler sanity (Q-04): train-benign scaled mean ≈ 0, std ≈ 1.
- No label column in `X` (ERD C3); `Label` never seen by scaler.